# Kaleido — Pipeline 3 (v3): Curriculum Assembly

**Job:** Read P2 enriched snapshot → assemble prep-unit rows with JSONB `sentences` + `practices` columns → write `prep_unit` table to Supabase.

**What this produces:**

| Output | Format | Purpose |
|--------|--------|---------|
| `prep_unit` | Supabase table | One row per learning unit — sentences JSONB + practices JSONB (12 items) |

**Key changes from v2:**
- `practices` is an **ordered array of 12 objects** (not a keyed dict)
- MCQ/Scramble/L1F practices have `versions[]` array (V1, V2)
- P0, L2F, L4W have **no versions** array
- Each unit appears **once** (not duplicated per tier)
- No `challenge` field (removed in UX redesign)
- Writes directly to Supabase (not data.json)

**Run cells top to bottom. Only Cell 2 requires your credentials.**

---
## Cell 1 — Install dependencies

In [1]:
!pip install supabase spacy --quiet
!python -m spacy download en_core_web_sm --quiet
print('✅ Dependencies ready')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 716.3 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 71.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
✅ Dependencies ready


---
## Cell 2 — Your Supabase credentials

In [2]:
SUPABASE_URL   = "https://upqgkokiemebdxhfdada.supabase.co"
SUPABASE_KEY   = "sb_publishable_YFvR_PWzN_seHZxg9SoJDw_73rhSoM3"

---
## Cell 3 — Connect to Supabase + load spaCy

In [3]:
from supabase import create_client
import spacy

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
nlp = spacy.load('en_core_web_sm')

print('✅ Supabase connected')
print('✅ spaCy loaded')

✅ Supabase connected
✅ spaCy loaded


---
## Cell 4 — Load P2 enriched snapshot from Google Drive

Downloads `p2_enriched_snapshot.json` — the file P2 uploaded in its final cell.

In [4]:
import json
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

auth.authenticate_user()
service = build('drive', 'v3')

FOLDER_ID = '1eOMwvWuJw_BsOaQXcgTKVLx2ykK77UXE'
FILE_NAME = 'p2_enriched_snapshot.json'

files = service.files().list(
    q=f"name='{FILE_NAME}' and '{FOLDER_ID}' in parents and trashed=false",
    fields='files(id)'
).execute().get('files', [])

if not files:
    raise FileNotFoundError(f'❌ {FILE_NAME} not found in Drive — run P2 first')

request = service.files().get_media(fileId=files[0]['id'])
with open(FILE_NAME, 'wb') as f:
    downloader = MediaIoBaseDownload(f, request)
    done = False
    while not done:
        _, done = downloader.next_chunk()

print(f'✅ Downloaded {FILE_NAME}')

with open(FILE_NAME, 'r', encoding='utf-8') as f:
    snapshot = json.load(f)

# Sanity checks
assert 'batch_id' in snapshot, '❌ Missing batch_id'
assert 'tier_50'  in snapshot, '❌ Missing tier_50'
assert 'tier_80'  in snapshot, '❌ Missing tier_80'

BATCH_ID = snapshot['batch_id']

print(f'batch_id      : {BATCH_ID}')
print(f'generated_at  : {snapshot.get("generated_at", "?")}')
print(f'tier_50 steps : {len(snapshot["tier_50"]["sequence"])}')
print(f'tier_80 steps : {len(snapshot["tier_80"]["sequence"])}')
print(f'tier_50 pool  : {len(snapshot["tier_50"]["pool"])} essays')
print(f'tier_80 pool  : {len(snapshot["tier_80"]["pool"])} essays')
print('\n✅ Snapshot loaded — ready for assembly')

✅ Downloaded p2_enriched_snapshot.json
batch_id      : 00b60e61-22d9-4513-8d08-a8f7c09e6c79
generated_at  : 2026-03-12T03:00:58.303884Z
tier_50 steps : 6
tier_80 steps : 14
tier_50 pool  : 53 essays
tier_80 pool  : 75 essays

✅ Snapshot loaded — ready for assembly


---
## Cell 5 — Define lookup tables

Two hardcoded dictionaries used by assembler functions.
(DIRECTION_LOOKUP is now in Supabase `direction_ref` table — fetched here.)

In [5]:
# ── STRUCTURE_LOOKUP (7 entries) ───────────────────────────────────────────
STRUCTURE_LOOKUP = {
    "structure_1": "4-Paragraph Discuss Both Views",
    "structure_2": "5-Paragraph Discuss Both Views + Opinion",
    "structure_3": "4-Paragraph Opinion-Led",
    "structure_4": "5-Paragraph Opinion + Counterargument",
    "structure_5": "4-Paragraph Advantages–Disadvantages",
    "structure_6": "4-Paragraph Problem–Solution",
    "structure_7": "5-Paragraph Causes–Effects–Solutions",
}
print(f'✅ STRUCTURE_LOOKUP — {len(STRUCTURE_LOOKUP)} entries')

# ── RHETORIC_LABEL_LOOKUP (33 entries) ────────────────────────────────────
RHETORIC_LABEL_LOOKUP = {
    "paraphrase_question":       "Introduction — restates the question in new words",
    "outline_statement":         "Introduction — previews the essay structure",
    "clear_thesis_statement":    "Introduction — states the writer's direct position",
    "topic_sentence":            "Body — introduces the paragraph's main claim",
    "explanation":               "Body — unpacks the causal mechanism",
    "example_1":                 "Body — first concrete evidence",
    "example_2":                 "Body — second concrete evidence",
    "mini_conclusion":           "Body — summarises the paragraph's logic",
    "link_to_thesis":            "Body — connects paragraph back to main argument",
    "opinion_topic_sentence":    "Opinion — opens with the writer's stance",
    "reason_1":                  "Opinion — first reason supporting the opinion",
    "reason_2":                  "Opinion — second reason supporting the opinion",
    "synthesis":                 "Opinion — combines ideas into a nuanced conclusion",
    "acknowledge_opposing_view": "Counterargument — introduces the opposing position",
    "concession":                "Counterargument — admits a valid point in opposition",
    "rebuttal":                  "Counterargument — refutes the opposing argument",
    "return_to_thesis":          "Counterargument — reasserts the writer's position",
    "problem_1":                 "Body — first problem identified",
    "problem_2":                 "Body — second problem identified",
    "solution_1":                "Body — first solution proposed",
    "solution_2":                "Body — second solution proposed",
    "cause_1":                   "Body — first cause identified",
    "cause_2":                   "Body — second cause identified",
    "effect_1":                  "Body — first effect described",
    "effect_2":                  "Body — second effect described",
    "advantage_1":               "Body — first advantage stated",
    "advantage_2":               "Body — second advantage stated",
    "disadvantage_1":            "Body — first disadvantage stated",
    "disadvantage_2":            "Body — second disadvantage stated",
    "opinion_statement":         "Conclusion — states the writer's final opinion",
    "restate_thesis":            "Conclusion — restates the main argument",
    "summary_statement":         "Conclusion — summarises the key points",
    "which_outweighs_opinion":   "Conclusion — declares which side is stronger",
}
print(f'✅ RHETORIC_LABEL_LOOKUP — {len(RHETORIC_LABEL_LOOKUP)} entries')

# ── DIRECTION_LOOKUP (from Supabase direction_ref table) ──────────────────
dref_rows = supabase.table('direction_ref').select('*').execute().data
DIRECTION_LOOKUP = {r['direction_id']: r['argument'] for r in dref_rows}
print(f'✅ DIRECTION_LOOKUP — {len(DIRECTION_LOOKUP)} entries loaded from Supabase')

✅ STRUCTURE_LOOKUP — 7 entries
✅ RHETORIC_LABEL_LOOKUP — 33 entries
✅ DIRECTION_LOOKUP — 43 entries loaded from Supabase


---
## Cell 6 — Utility functions

Five reusable helpers used by the assembler functions in Cell 7.

In [6]:
import random

# ── POS tagger ──────────────────────────────────────────────────────────────
_POS_MAP = {
    'NOUN': 'NOUN', 'PROPN': 'NOUN',
    'VERB': 'VERB', 'AUX':   'VERB',
    'ADJ':  'ADJ',  'ADV':   'ADV',
}

def pos_tag(phrase):
    """Returns POS of a phrase: NOUN, VERB, ADJ, ADV, or PHRASE."""
    doc = nlp(phrase)
    root_tokens = [t for t in doc if t.dep_ == 'ROOT']
    if not root_tokens:
        return 'PHRASE'
    return _POS_MAP.get(root_tokens[0].pos_, 'PHRASE')


# ── Sentence chunker for L2S ────────────────────────────────────────────────
def chunk_sentence(canonical_text, lexical_items):
    """
    Splits a sentence into chunks for L2S scramble.
    Lexical phrases stay as one chip; other words are individual chips.
    """
    phrases = [
        li['phrase'] if isinstance(li, dict) else li
        for li in lexical_items
    ]
    sorted_phrases = sorted(phrases, key=len, reverse=True)

    chunks = []
    remaining = canonical_text

    while remaining:
        matched = False
        for phrase in sorted_phrases:
            if remaining.startswith(phrase):
                chunks.append({'text': phrase, 'is_lexical': True})
                remaining = remaining[len(phrase):].lstrip(' ')
                matched = True
                break
        if not matched:
            word, _, remaining = remaining.partition(' ')
            chunks.append({'text': word, 'is_lexical': False})
    return chunks


# ── Distractor pool builder ─────────────────────────────────────────────────
def build_distractor_pools(step_n, tier_data):
    """
    Builds 4 priority buckets of candidate sentences for step N.
    Pool 1: challenge (always empty post-redesign)
    Pool 2: next-step essays
    Pool 3: previous-step essays
    Pool 4: all remaining tier pool essays
    """
    sequence = tier_data['sequence']
    pool     = tier_data['pool']
    step     = sequence[step_n]

    used_essay_ids = set()

    # Pool 1: challenge (always empty)
    pool1 = []
    if step.get('challenge'):
        pool1 = step['challenge']['sentences']
        used_essay_ids.add(step['challenge']['essay_id'])
    if step.get('model'):
        used_essay_ids.add(step['model']['essay_id'])

    # Pool 2: next-step essays
    pool2 = []
    if step_n + 1 < len(sequence):
        next_step = sequence[step_n + 1]
        if next_step.get('model'):
            pool2 += next_step['model']['sentences']
            used_essay_ids.add(next_step['model']['essay_id'])

    # Pool 3: previous-step essays
    pool3 = []
    if step_n - 1 >= 0:
        prev_step = sequence[step_n - 1]
        if prev_step.get('model'):
            pool3 += prev_step['model']['sentences']
            used_essay_ids.add(prev_step['model']['essay_id'])

    # Pool 4: all remaining
    pool4 = []
    for essay in pool:
        if essay['essay_id'] not in used_essay_ids:
            pool4 += essay['sentences']

    return [pool1, pool2, pool3, pool4]


# ── Distractor selector ─────────────────────────────────────────────────────
def get_distractors(tag, pools, n=4, exclude_ids=None):
    """Picks n distractor sentences matching rhetoric_tag, with Priority 5 fallback."""
    exclude_ids = exclude_ids or set()
    candidates  = []
    seen_ids    = set(exclude_ids)

    for pool in pools:
        for sentence in pool:
            sid = sentence['sentence_id']
            if sid not in seen_ids and sentence.get('rhetoric_tag') == tag:
                candidates.append(sentence)
                seen_ids.add(sid)
        if len(candidates) >= n:
            return candidates[:n]

    # Priority 5: relax tag constraint
    if len(candidates) < n:
        for sentence in pools[3]:
            sid = sentence['sentence_id']
            if sid not in seen_ids:
                candidates.append(sentence)
                seen_ids.add(sid)
            if len(candidates) >= n:
                break

    return candidates[:n]


# ── MCQ option shuffler ─────────────────────────────────────────────────────
def shuffle_options(correct_text, distractors):
    """Packages correct + distractors into shuffled 5-option MCQ."""
    distractor_texts = [
        d['canonical_text'] if isinstance(d, dict) else d
        for d in distractors[:4]
    ]
    options = [correct_text] + distractor_texts
    random.shuffle(options)
    return {
        'options':       options,
        'correct_index': options.index(correct_text),
    }


print('✅ All 5 utility functions ready')

✅ All 5 utility functions ready


---
## Cell 7 — All 12 assembler functions

Each function assembles one practice type. The output shapes match the
dev brief §5.4 spec. Key changes from v2:
- MCQ/Scramble/L1F return `versions[]` array
- L2F returns flat questions (no versions)
- P0 and L4W return simple dicts (no versions)

In [7]:
_PARA_ORDER      = ['introduction', 'body_1', 'body_2', 'body_3', 'conclusion']
_BODY_PARAS      = ['body_1', 'body_2', 'body_3']
_PARA_ABBREV     = {'introduction': 'intro', 'body_1': 'B1', 'body_2': 'B2', 'body_3': 'B3', 'conclusion': 'concl'}


# ═══════════════════════════════════════════════════════════════════════════════
# P0 — Cold Write
# ═══════════════════════════════════════════════════════════════════════════════
def assemble_P0(step):
    return {
        "practice_code": "P0",
        "question": step['model']['question'],
        "prompt":   "Write a complete IELTS Task 2 essay in response to this question:",
    }


# ═══════════════════════════════════════════════════════════════════════════════
# L4W — Essay Write (unit gate)
# ═══════════════════════════════════════════════════════════════════════════════
def assemble_L4W(step):
    return {"practice_code": "L4W"}


# ═══════════════════════════════════════════════════════════════════════════════
# L1S — Lexical Scramble
# ═══════════════════════════════════════════════════════════════════════════════
def assemble_L1S(step):
    questions_v1 = []
    questions_v2 = []
    q_num = 0

    for s in step['model']['sentences']:
        for li in s.get('lexical_items', []):
            phrase = li['phrase']
            words  = phrase.split()
            if len(words) < 2:
                continue
            q_num += 1
            q_id = f"L1S-{q_num}"

            # V1 and V2 get different shuffles
            chips_v1 = words.copy(); random.shuffle(chips_v1)
            chips_v2 = words.copy(); random.shuffle(chips_v2)

            base = {"id": q_id, "phrase": phrase, "answer": phrase}
            questions_v1.append({**base, "chips": chips_v1})
            questions_v2.append({**base, "chips": chips_v2})

    return {
        "practice_code": "L1S",
        "versions": [
            {"version": "V1", "questions": questions_v1},
            {"version": "V2", "questions": questions_v2},
        ]
    }


# ═══════════════════════════════════════════════════════════════════════════════
# L2S — Sentence Scramble
# ═══════════════════════════════════════════════════════════════════════════════
def assemble_L2S(step):
    model = step['model']
    questions_v1 = []
    questions_v2 = []

    for s in model['sentences']:
        if s['paragraph_type'] not in _BODY_PARAS:
            continue
        chunks = chunk_sentence(s['canonical_text'], s.get('lexical_items', []))

        hint = s['syntax_items'][0] if s.get('syntax_items') else RHETORIC_LABEL_LOOKUP.get(s['rhetoric_tag'], s['rhetoric_tag'])
        q_id = f"L2S-{s['paragraph_type']}-S{s['order']}"
        base = {"id": q_id, "paragraph": s['paragraph_type'], "sentence_order": s['order'], "hint": hint, "answer": s['canonical_text']}

        c1 = chunks.copy(); random.shuffle(c1)
        c2 = chunks.copy(); random.shuffle(c2)
        questions_v1.append({**base, "chunks": c1})
        questions_v2.append({**base, "chunks": c2})

    return {
        "practice_code": "L2S",
        "versions": [
            {"version": "V1", "questions": questions_v1},
            {"version": "V2", "questions": questions_v2},
        ]
    }


# ═══════════════════════════════════════════════════════════════════════════════
# L3S — Paragraph Scramble
# ═══════════════════════════════════════════════════════════════════════════════
def assemble_L3S(step):
    model = step['model']
    questions_v1 = []
    questions_v2 = []

    for para in _BODY_PARAS:
        para_sents = sorted(
            [s for s in model['sentences'] if s['paragraph_type'] == para],
            key=lambda s: s['order']
        )
        if not para_sents:
            continue

        hint = ' → '.join(
            RHETORIC_LABEL_LOOKUP.get(s['rhetoric_tag'], s['rhetoric_tag'])
            for s in para_sents
        )
        answer_order = [s['sentence_id'] for s in para_sents]
        display = [{'sentence_id': s['sentence_id'], 'text': s['canonical_text']} for s in para_sents]

        d1 = display.copy(); random.shuffle(d1)
        d2 = display.copy(); random.shuffle(d2)

        base = {"id": f"L3S-{para}", "paragraph": para, "hint": hint, "answer_order": answer_order}
        questions_v1.append({**base, "sentences": d1})
        questions_v2.append({**base, "sentences": d2})

    return {
        "practice_code": "L3S",
        "versions": [
            {"version": "V1", "questions": questions_v1},
            {"version": "V2", "questions": questions_v2},
        ]
    }


# ═══════════════════════════════════════════════════════════════════════════════
# L4S — Essay Scramble
# ═══════════════════════════════════════════════════════════════════════════════
def assemble_L4S(step):
    model = step['model']
    para_openers = {}
    for s in model['sentences']:
        if s['order'] == 1:
            para_openers[s['paragraph_type']] = s['canonical_text']

    ordered = [{'paragraph': p, 'text': para_openers[p]} for p in _PARA_ORDER if p in para_openers]
    answer_order = [o['paragraph'] for o in ordered]
    structure_label = STRUCTURE_LOOKUP.get(model['structure_type'], model['structure_type'])

    s1 = ordered.copy(); random.shuffle(s1)
    s2 = ordered.copy(); random.shuffle(s2)

    base = {"hint": structure_label, "answer_order": answer_order}
    return {
        "practice_code": "L4S",
        "versions": [
            {"version": "V1", **base, "openers": s1},
            {"version": "V2", **base, "openers": s2},
        ]
    }


# ═══════════════════════════════════════════════════════════════════════════════
# L1F — Phrase Fill
# ═══════════════════════════════════════════════════════════════════════════════
def assemble_L1F(step):
    model = step['model']
    questions_v1 = []
    questions_v2 = []

    for para in _PARA_ORDER:
        para_sents = sorted(
            [s for s in model['sentences'] if s['paragraph_type'] == para],
            key=lambda s: s['order']
        )
        if not para_sents:
            continue

        for target_s in para_sents:
            lex_items = target_s.get('lexical_items', [])
            if not lex_items:
                continue

            phrases = [li['phrase'] for li in lex_items]
            sorted_phrases = sorted(phrases, key=len, reverse=True)
            phrase_to_li = {li['phrase']: li for li in lex_items}

            # Build parts + blanks
            parts = []
            blanks = []
            blank_index = 0
            current_text = ""
            remaining = target_s['canonical_text']

            while remaining:
                matched = False
                for phrase in sorted_phrases:
                    if remaining.startswith(phrase):
                        if current_text:
                            parts.append({"type": "text", "text": current_text})
                            current_text = ""
                        li = phrase_to_li[phrase]
                        parts.append({"type": "blank", "blank_index": blank_index})
                        blanks.append({
                            "index":           blank_index,
                            "answer":          phrase,
                            "pos_hint":        pos_tag(phrase),
                            "similar_phrases": li.get('similar_phrases', [])
                        })
                        blank_index += 1
                        remaining = remaining[len(phrase):]
                        matched = True
                        break
                if not matched:
                    current_text += remaining[0]
                    remaining = remaining[1:]

            if current_text:
                parts.append({"type": "text", "text": current_text})

            # Context: full paragraph, target has parts
            context = []
            for s in para_sents:
                if s['sentence_id'] == target_s['sentence_id']:
                    context.append({"order": s['order'], "parts": parts, "isTarget": True})
                else:
                    context.append({"order": s['order'], "text": s['canonical_text'], "isTarget": False})

            abbrev = _PARA_ABBREV.get(para, para)
            q_id = f"L1F-{abbrev}-S{target_s['order']}"

            q = {
                "id": q_id,
                "paragraph": para,
                "sentence_order": target_s['order'],
                "rhetoric_tag": target_s.get('rhetoric_tag', ''),
                "context": context,
                "blanks": blanks,
                "prompt": "Fill in the missing phrases. Use the part of speech hint and similar phrases below each blank to understand what kind of phrase is missing."
            }
            questions_v1.append(q)
            questions_v2.append(q)  # L1F V1 = V2 (same blanks, different from MCQ)

    return {
        "practice_code": "L1F",
        "versions": [
            {"version": "V1", "questions": questions_v1},
            {"version": "V2", "questions": questions_v2},
        ]
    }


# ═══════════════════════════════════════════════════════════════════════════════
# L2F — Sentence Fill (NO versions — no retry)
# ═══════════════════════════════════════════════════════════════════════════════
def assemble_L2F(step):
    model = step['model']
    questions = []

    for para in _PARA_ORDER:
        para_sents = sorted(
            [s for s in model['sentences'] if s['paragraph_type'] == para],
            key=lambda s: s['order']
        )
        if not para_sents:
            continue

        for target_s in para_sents:
            context = []
            for s in para_sents:
                if s['sentence_id'] == target_s['sentence_id']:
                    context.append({"order": s['order'], "text": "___", "isTarget": True})
                else:
                    context.append({"order": s['order'], "text": s['canonical_text'], "isTarget": False})

            rhetoric_tag  = target_s.get('rhetoric_tag', '')
            rhetoric_hint = RHETORIC_LABEL_LOOKUP.get(rhetoric_tag, rhetoric_tag)
            hint_sentences = target_s.get('hint_sentences', [])

            abbrev = _PARA_ABBREV.get(para, para)
            q_id = f"L2F-{abbrev}-S{target_s['order']}"

            questions.append({
                "id":             q_id,
                "paragraph":      para,
                "sentence_order": target_s['order'],
                "rhetoric_tag":   rhetoric_tag,
                "rhetoric_hint":  rhetoric_hint,
                "hint_sentences": hint_sentences,
                "context":        context,
                "prompt":         "Fill in the missing sentence. Use the hint sentences and structure label below to understand what kind of sentence belongs here."
            })

    # L2F has NO versions array — no retry
    return {
        "practice_code": "L2F",
        "questions": questions,
    }


print('✅ Assemblers ready: P0, L4W, L1S, L2S, L3S, L4S, L1F, L2F')

✅ Assemblers ready: P0, L4W, L1S, L2S, L3S, L4S, L1F, L2F


---
## Cell 8 — MCQ assembler functions (L1M, L2M, L3M, L4M)

MCQ practices have V1/V2 versions with different distractor priority ordering.

In [8]:
# ── MCQ helpers ──────────────────────────────────────────────────────────────

def _pools_v2(pools):
    """Swap pool[0] and pool[1] for V2 priority ordering."""
    return [pools[1], pools[0], pools[2], pools[3]]


def _essay_priority_pool(step_n, tier_data, version='V1'):
    """Returns priority-ordered list of ESSAY objects for L4M distractors."""
    sequence = tier_data['sequence']
    pool     = tier_data['pool']
    step     = sequence[step_n]

    used_ids = set()
    ordered  = []

    if step.get('model'):
        used_ids.add(step['model']['essay_id'])

    if version == 'V1':
        ch = step.get('challenge')
        if ch and ch['essay_id'] not in used_ids:
            ordered.append(ch); used_ids.add(ch['essay_id'])
        if step_n + 1 < len(sequence):
            for key in ['model', 'challenge']:
                e = sequence[step_n + 1].get(key)
                if e and e['essay_id'] not in used_ids:
                    ordered.append(e); used_ids.add(e['essay_id'])
    else:
        if step_n + 1 < len(sequence):
            for key in ['model', 'challenge']:
                e = sequence[step_n + 1].get(key)
                if e and e['essay_id'] not in used_ids:
                    ordered.append(e); used_ids.add(e['essay_id'])
        ch = step.get('challenge')
        if ch and ch['essay_id'] not in used_ids:
            ordered.append(ch); used_ids.add(ch['essay_id'])

    for essay in pool:
        if essay['essay_id'] not in used_ids:
            ordered.append(essay); used_ids.add(essay['essay_id'])

    return ordered


def _direction_distractors(direction_tag, pools, n=4):
    """Finds n direction argument texts from pool where direction_tag differs."""
    dist_texts = []
    seen_tags  = {direction_tag}
    for pool in pools:
        for s in pool:
            dtag = s.get('direction_tag')
            if dtag and dtag not in seen_tags:
                dtext = DIRECTION_LOOKUP.get(dtag)
                if dtext and dtext not in dist_texts:
                    dist_texts.append(dtext)
                    seen_tags.add(dtag)
        if len(dist_texts) >= n:
            break
    return dist_texts[:n]


# ═══════════════════════════════════════════════════════════════════════════════
# L1M — Lexical MCQ
# ═══════════════════════════════════════════════════════════════════════════════
def assemble_L1M(step_n, tier_data):
    step  = tier_data['sequence'][step_n]
    model = step['model']

    model_by_pos = {}
    for s in model['sentences']:
        for li in s.get('lexical_items', []):
            pos = pos_tag(li['phrase'])
            model_by_pos.setdefault(pos, []).append(li['phrase'])

    def pick_phrase_distractors(phrase, pos, n=4):
        seen = {phrase}; result = []
        for p in model_by_pos.get(pos, []):
            if p not in seen: result.append(p); seen.add(p)
        for other_pos, phrases in model_by_pos.items():
            if other_pos == pos: continue
            for p in phrases:
                if p not in seen: result.append(p); seen.add(p)
        return result[:n]

    questions_v1 = []
    questions_v2 = []
    q_num = 0
    seen_phrases = set()

    for s in model['sentences']:
        for li in s.get('lexical_items', []):
            phrase = li['phrase']
            if phrase in seen_phrases: continue
            seen_phrases.add(phrase)
            q_num += 1
            pos = pos_tag(phrase)
            blank_sentence = s['canonical_text'].replace(phrase, '____', 1)
            distractors = pick_phrase_distractors(phrase, pos)

            base = {"id": f"L1M-{q_num}", "sentence": s['canonical_text'], "blank_sentence": blank_sentence, "prompt": "All options appear in the essay — which phrase belongs here?", "pos_hint": pos}
            questions_v1.append({**base, **shuffle_options(phrase, distractors)})
            questions_v2.append({**base, **shuffle_options(phrase, distractors)})

    return {
        "practice_code": "L1M",
        "versions": [
            {"version": "V1", "questions": questions_v1},
            {"version": "V2", "questions": questions_v2},
        ]
    }


# ═══════════════════════════════════════════════════════════════════════════════
# L2M — Sentence MCQ
# ═══════════════════════════════════════════════════════════════════════════════
def assemble_L2M(step_n, tier_data):
    step  = tier_data['sequence'][step_n]
    model = step['model']
    pools    = build_distractor_pools(step_n, tier_data)
    pools_v1 = pools
    pools_v2 = _pools_v2(pools)

    structure_tags = list(dict.fromkeys(s['rhetoric_tag'] for s in model['sentences']))

    questions_v1 = []
    questions_v2 = []
    q_num = 0

    for s in model['sentences']:
        q_num += 1

        # Template A: Rhetoric role
        correct_label = RHETORIC_LABEL_LOOKUP.get(s['rhetoric_tag'], s['rhetoric_tag'])
        other_labels  = [RHETORIC_LABEL_LOOKUP.get(t, t) for t in structure_tags if t != s['rhetoric_tag']]

        questions_v1.append({"id": f"L2M-{q_num}-A", "type": "rhetoric", "sentence": s['canonical_text'], "prompt": "What structural role does this sentence play?", **shuffle_options(correct_label, other_labels[:4])})
        questions_v2.append({"id": f"L2M-{q_num}-A", "type": "rhetoric", "sentence": s['canonical_text'], "prompt": "What structural role does this sentence play?", **shuffle_options(correct_label, other_labels[:4])})

        # Template B: PoV (only if direction_tag present)
        dtag = s.get('direction_tag')
        if dtag:
            correct_dir = DIRECTION_LOOKUP.get(dtag)
            if correct_dir:
                v1_dist = _direction_distractors(dtag, pools_v1)
                v2_dist = _direction_distractors(dtag, pools_v2)
                questions_v1.append({"id": f"L2M-{q_num}-B", "type": "direction", "sentence": s['canonical_text'], "prompt": "Which PoV is this sentence supporting?", **shuffle_options(correct_dir, v1_dist)})
                questions_v2.append({"id": f"L2M-{q_num}-B", "type": "direction", "sentence": s['canonical_text'], "prompt": "Which PoV is this sentence supporting?", **shuffle_options(correct_dir, v2_dist)})

    return {
        "practice_code": "L2M",
        "versions": [
            {"version": "V1", "questions": questions_v1},
            {"version": "V2", "questions": questions_v2},
        ]
    }


# ═══════════════════════════════════════════════════════════════════════════════
# L3M — Paragraph MCQ
# ═══════════════════════════════════════════════════════════════════════════════
def assemble_L3M(step_n, tier_data):
    step  = tier_data['sequence'][step_n]
    model = step['model']
    pools    = build_distractor_pools(step_n, tier_data)
    pools_v1 = pools
    pools_v2 = _pools_v2(pools)

    questions_v1 = []
    questions_v2 = []

    for para in _BODY_PARAS:
        para_sents = sorted(
            [s for s in model['sentences'] if s['paragraph_type'] == para],
            key=lambda s: s['order']
        )
        if not para_sents: continue

        # Template A: Topic sentence
        topic_sents = [s for s in para_sents if s['rhetoric_tag'] == 'topic_sentence']
        if topic_sents:
            topic = topic_sents[0]
            context = [s['canonical_text'] for s in para_sents if s['sentence_id'] != topic['sentence_id']]
            exclude = {topic['sentence_id']}
            v1_dist = get_distractors('topic_sentence', pools_v1, n=4, exclude_ids=exclude)
            v2_dist = get_distractors('topic_sentence', pools_v2, n=4, exclude_ids=exclude)

            base = {"id": f"L3M-{para}-A", "type": "topic_sentence", "paragraph": para, "prompt": "What is the topic sentence that these supporting sentences belong to?", "context": context}
            questions_v1.append({**base, **shuffle_options(topic['canonical_text'], [d['canonical_text'] for d in v1_dist])})
            questions_v2.append({**base, **shuffle_options(topic['canonical_text'], [d['canonical_text'] for d in v2_dist])})

        # Template C: PoV per paragraph
        primary_dtag = next((s.get('direction_tag') for s in para_sents if s.get('direction_tag')), None)
        if primary_dtag:
            correct_dir = DIRECTION_LOOKUP.get(primary_dtag)
            if correct_dir:
                full_text = ' '.join(s['canonical_text'] for s in para_sents)
                v1_dist = _direction_distractors(primary_dtag, pools_v1)
                v2_dist = _direction_distractors(primary_dtag, pools_v2)

                base = {"id": f"L3M-{para}-C", "type": "direction", "paragraph": para, "prompt": "Which PoV is this paragraph developing?", "context": full_text}
                questions_v1.append({**base, **shuffle_options(correct_dir, v1_dist)})
                questions_v2.append({**base, **shuffle_options(correct_dir, v2_dist)})

    return {
        "practice_code": "L3M",
        "versions": [
            {"version": "V1", "questions": questions_v1},
            {"version": "V2", "questions": questions_v2},
        ]
    }


# ═══════════════════════════════════════════════════════════════════════════════
# L4M — Essay MCQ (3 fixed questions)
# ═══════════════════════════════════════════════════════════════════════════════
def assemble_L4M(step_n, tier_data):
    step  = tier_data['sequence'][step_n]
    model = step['model']

    correct_structure = STRUCTURE_LOOKUP.get(model['structure_type'], model['structure_type'])
    dist_structures   = [v for v in STRUCTURE_LOOKUP.values() if v != correct_structure][:4]

    correct_q = model['question']

    # Build correct PoV string
    model_dir_args = []
    seen_dtags     = set()
    for s in model['sentences']:
        dtag = s.get('direction_tag')
        if dtag and dtag not in seen_dtags:
            arg = DIRECTION_LOOKUP.get(dtag)
            if arg: model_dir_args.append(arg)
            seen_dtags.add(dtag)
    correct_pov = ' / '.join(model_dir_args) if model_dir_args else '—'

    version_data = {}
    for version in ['V1', 'V2']:
        essay_pool = _essay_priority_pool(step_n, tier_data, version)

        # L4M-1 distractors: questions from pool
        dist_qs = []; seen_qs = {correct_q}
        for essay in essay_pool:
            q = essay.get('question', '')
            if q and q not in seen_qs: dist_qs.append(q); seen_qs.add(q)
            if len(dist_qs) >= 4: break

        # L4M-2 distractors: direction pairs from pool
        dist_povs = []; seen_povs = {correct_pov}
        for essay in essay_pool:
            args = []; seen_tags = set()
            for s in essay.get('sentences', []):
                dtag = s.get('direction_tag')
                if dtag and dtag not in seen_tags:
                    arg = DIRECTION_LOOKUP.get(dtag)
                    if arg: args.append(arg)
                    seen_tags.add(dtag)
            pov_str = ' / '.join(args) if args else None
            if pov_str and pov_str not in seen_povs: dist_povs.append(pov_str); seen_povs.add(pov_str)
            if len(dist_povs) >= 4: break

        version_data[version] = {
            'q1': shuffle_options(correct_q, dist_qs),
            'q2': shuffle_options(correct_pov, dist_povs),
            'q3': shuffle_options(correct_structure, dist_structures),
        }

    questions_v1 = [
        {"id": "L4M-1", "prompt": "Which IELTS question does this essay respond to?", **version_data['V1']['q1']},
        {"id": "L4M-2", "prompt": "Which pair of PoVs does this essay develop?", **version_data['V1']['q2']},
        {"id": "L4M-3", "prompt": "Which essay structure is used here?", **version_data['V1']['q3']},
    ]
    questions_v2 = [
        {"id": "L4M-1", "prompt": "Which IELTS question does this essay respond to?", **version_data['V2']['q1']},
        {"id": "L4M-2", "prompt": "Which pair of PoVs does this essay develop?", **version_data['V2']['q2']},
        {"id": "L4M-3", "prompt": "Which essay structure is used here?", **version_data['V2']['q3']},
    ]

    return {
        "practice_code": "L4M",
        "versions": [
            {"version": "V1", "questions": questions_v1},
            {"version": "V2", "questions": questions_v2},
        ]
    }


print('✅ MCQ assemblers ready: L1M, L2M, L3M, L4M')

✅ MCQ assemblers ready: L1M, L2M, L3M, L4M


---
## Cell 9 — Assemble `sentences` JSONB helper

Strips internal fields from essay sentences. The `sentences` JSONB on `prep_unit`
is for display only (peek modal, L4W reference panels). No `similar_phrases` or
`hint_sentences` — those live inside `practices`.

In [9]:
def build_sentences_jsonb(model_essay):
    """
    Build the sentences JSONB for a prep_unit row.
    Strips hint_sentences, similar_phrases — those are consumed by practices.
    Adds rhetoric_label and pos on lexical items.
    """
    sentences = []
    for s in model_essay['sentences']:
        sentences.append({
            'sentence_id':    s['sentence_id'],
            'paragraph_type': s['paragraph_type'],
            'order':          s['order'],
            'canonical_text': s['canonical_text'],
            'rhetoric_tag':   s['rhetoric_tag'],
            'direction_tag':  s.get('direction_tag'),
            'rhetoric_label': RHETORIC_LABEL_LOOKUP.get(s.get('rhetoric_tag', ''), s.get('rhetoric_tag', '')),
            'lexical_items':  [
                {'phrase': li['phrase'], 'pos': pos_tag(li['phrase'])}
                for li in s.get('lexical_items', [])
            ],
            'syntax_items':   s.get('syntax_items', []),
        })
    return sentences


print('✅ build_sentences_jsonb ready')

✅ build_sentences_jsonb ready


---
## Cell 10 — Main assembly loop

Iterates over all tiers and steps, assembles prep_unit rows.
**Deduplication:** Each model essay gets ONE prep_unit row, even if it appears
in both tier_50 and tier_80. The tier assignment lives in `tier_unit_sequence`.

In [10]:
# Track which unit_ids we've already assembled (deduplication)
assembled_units = {}  # unit_id -> prep_unit row

print('Assembling prep-units...')

for tier_name in ['tier_50', 'tier_80']:
    tier_data = snapshot[tier_name]
    n_steps   = len(tier_data['sequence'])
    print(f'  {tier_name}: {n_steps} steps')

    for step_n in range(n_steps):
        step  = tier_data['sequence'][step_n]
        model = step['model']
        unit_id = model['essay_id']

        # Skip if already assembled (shared unit between tiers)
        if unit_id in assembled_units:
            continue

        # Build the 12-practice ordered array
        practices = [
            assemble_P0(step),                       # P0
            assemble_L4M(step_n, tier_data),         # L4M
            assemble_L3M(step_n, tier_data),         # L3M
            assemble_L2M(step_n, tier_data),         # L2M
            assemble_L1M(step_n, tier_data),         # L1M
            assemble_L1S(step),                      # L1S
            assemble_L2S(step),                      # L2S
            assemble_L3S(step),                      # L3S
            assemble_L4S(step),                      # L4S
            assemble_L1F(step),                      # L1F
            assemble_L2F(step),                      # L2F
            assemble_L4W(step),                      # L4W
        ]

        # Build sentences JSONB
        sentences_jsonb = build_sentences_jsonb(model)

        # Enrich directions for display
        directions = [
            {"id": d, "argument": DIRECTION_LOOKUP.get(d, d)}
            for d in model.get('directions', [])
        ]

        assembled_units[unit_id] = {
            'unit_id':        unit_id,
            'batch_id':       BATCH_ID,
            'question':       model['question'],
            'structure_type': model['structure_type'],
            'sentences':      sentences_jsonb,
            'practices':      practices,
        }

print(f'\nTotal unique prep-units: {len(assembled_units)}')
print(f'Practice codes per unit: {[p["practice_code"] for p in list(assembled_units.values())[0]["practices"]]}')
print('✅ Assembly complete')

Assembling prep-units...
  tier_50: 6 steps
  tier_80: 14 steps

Total unique prep-units: 14
Practice codes per unit: ['P0', 'L4M', 'L3M', 'L2M', 'L1M', 'L1S', 'L2S', 'L3S', 'L4S', 'L1F', 'L2F', 'L4W']
✅ Assembly complete


---
## Cell 11 — Write `prep_unit` to Supabase

```sql
-- Reference: create this table in Supabase dashboard first
CREATE TABLE prep_unit (
    unit_id        UUID PRIMARY KEY,
    batch_id       UUID NOT NULL,
    question       TEXT NOT NULL,
    structure_type TEXT NOT NULL,
    sentences      JSONB NOT NULL,
    practices      JSONB NOT NULL
);
```

In [11]:
import json

# Build rows — JSONB columns need to be JSON strings for Supabase
prep_unit_rows = []
for unit_id, unit in assembled_units.items():
    prep_unit_rows.append({
        'unit_id':        unit['unit_id'],
        'batch_id':       unit['batch_id'],
        'question':       unit['question'],
        'structure_type': unit['structure_type'],
        'sentences':      json.dumps(unit['sentences']),
        'practices':      json.dumps(unit['practices']),
    })

print(f'Rows to insert: {len(prep_unit_rows)}')

# Insert in batches of 10 (JSONB rows can be large)
batch_size = 10
for i in range(0, len(prep_unit_rows), batch_size):
    batch = prep_unit_rows[i:i + batch_size]
    resp = supabase.table('prep_unit').upsert(batch).execute()
    print(f'  Batch {i//batch_size + 1}: wrote {len(resp.data)} rows')

print(f'\n✅ Wrote {len(prep_unit_rows)} rows to prep_unit')

Rows to insert: 14
  Batch 1: wrote 10 rows
  Batch 2: wrote 4 rows

✅ Wrote 14 rows to prep_unit


---
## Cell 12 — Save data.json to Google Drive (backup)

Saves the complete output as a JSON file for inspection and backup.
The primary output is already in Supabase — this is just a readable copy.

In [12]:
import json, os

# Build the complete data.json shape
data_json = {
    "batch_id": BATCH_ID,
    "prep_units": list(assembled_units.values()),
}

OUTPUT_FILE = 'data.json'
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(data_json, f, indent=2, ensure_ascii=False)

size_mb = os.path.getsize(OUTPUT_FILE) / 1024 / 1024
print(f'✅ Saved {OUTPUT_FILE} ({size_mb:.2f} MB)')
print(f'   prep_units: {len(data_json["prep_units"])}')

# Upload to Google Drive
from googleapiclient.http import MediaFileUpload

UPLOAD_NAME = 'data.json'
existing = service.files().list(
    q=f"name='{UPLOAD_NAME}' and '{FOLDER_ID}' in parents and trashed=false",
    fields='files(id)'
).execute().get('files', [])

media = MediaFileUpload(OUTPUT_FILE, mimetype='application/json', resumable=False)
if existing:
    service.files().update(fileId=existing[0]['id'], media_body=media).execute()
    print(f'♻️  Replaced existing {UPLOAD_NAME} in Drive')
else:
    meta = {'name': UPLOAD_NAME, 'parents': [FOLDER_ID]}
    service.files().create(body=meta, media_body=media, fields='id').execute()
    print(f'✅ Uploaded {UPLOAD_NAME} to Drive')

✅ Saved data.json (3.87 MB)
   prep_units: 14
✅ Uploaded data.json to Drive


---
## Cell 13 — Verification checklist

Validates prep_unit rows in Supabase and cross-checks with P2 output.

In [13]:
passes   = []
failures = []

def check(label, condition, detail=''):
    if condition:
        passes.append(f'  ✅  {label}')
    else:
        failures.append(f'  ❌  {label}' + (f'\n       → {detail}' if detail else ''))

EXPECTED_PRACTICE_CODES = ['P0', 'L4M', 'L3M', 'L2M', 'L1M', 'L1S', 'L2S', 'L3S', 'L4S', 'L1F', 'L2F', 'L4W']
VERSIONED_CODES   = {'L4M', 'L3M', 'L2M', 'L1M', 'L1S', 'L2S', 'L3S', 'L4S', 'L1F'}
UNVERSIONED_CODES = {'P0', 'L2F', 'L4W'}

# ── Check Supabase rows ───────────────────────────────────────────────────
pu_check = supabase.table('prep_unit').select('unit_id, batch_id, practices').eq('batch_id', BATCH_ID).execute()
check('prep_unit has rows', len(pu_check.data) > 0, f'Got {len(pu_check.data)}')
check('prep_unit count matches', len(pu_check.data) == len(assembled_units), f'Expected {len(assembled_units)}, got {len(pu_check.data)}')

# ── Check practice structure ───────────────────────────────────────────────
for row in pu_check.data[:5]:  # spot-check first 5
    practices = json.loads(row['practices']) if isinstance(row['practices'], str) else row['practices']
    uid = row['unit_id'][:8]

    check(f'{uid}… has 12 practices', len(practices) == 12, f'Got {len(practices)}')

    codes = [p.get('practice_code') for p in practices]
    check(f'{uid}… practice codes match spec', codes == EXPECTED_PRACTICE_CODES, f'Got {codes}')

    for p in practices:
        code_name = p.get('practice_code', '?')
        if code_name in VERSIONED_CODES:
            check(f'{uid}… {code_name} has versions[]', 'versions' in p, f'Missing versions on {code_name}')
            if 'versions' in p:
                v_labels = [v.get('version') for v in p['versions']]
                check(f'{uid}… {code_name} versions are V1,V2', v_labels == ['V1', 'V2'], f'Got {v_labels}')
        elif code_name in UNVERSIONED_CODES:
            check(f'{uid}… {code_name} has NO versions', 'versions' not in p, f'{code_name} should not have versions')

# ── Cross-check with tier_unit_sequence ────────────────────────────────────
tus_data = supabase.table('tier_unit_sequence').select('unit_id').eq('batch_id', BATCH_ID).execute()
tus_unit_ids = {r['unit_id'] for r in tus_data.data}
pu_unit_ids  = {r['unit_id'] for r in pu_check.data}
missing = tus_unit_ids - pu_unit_ids
check('All tier_unit_sequence unit_ids have prep_unit rows', len(missing) == 0, f'Missing: {list(missing)[:3]}')

# ── Print results ──────────────────────────────────────────────────────────
print('=' * 60)
print('PIPELINE 3 v3 — VERIFICATION')
print('=' * 60)
for msg in passes:
    print(msg)
for msg in failures:
    print(msg)
print('=' * 60)
print(f'  {len(passes)} passed   {len(failures)} failed')

if not failures:
    print('\n🎉 Pipeline 3 complete — all checks passed.')
    print(f'   batch_id: {BATCH_ID}')
    print(f'   prep_units: {len(assembled_units)}')
    print('   The School can now ingest this data.')
else:
    print('\n🚨 Fix failures above before proceeding.')

PIPELINE 3 v3 — VERIFICATION
  ✅  prep_unit has rows
  ✅  prep_unit count matches
  ✅  5e9d3c6f… has 12 practices
  ✅  5e9d3c6f… practice codes match spec
  ✅  5e9d3c6f… P0 has NO versions
  ✅  5e9d3c6f… L4M has versions[]
  ✅  5e9d3c6f… L4M versions are V1,V2
  ✅  5e9d3c6f… L3M has versions[]
  ✅  5e9d3c6f… L3M versions are V1,V2
  ✅  5e9d3c6f… L2M has versions[]
  ✅  5e9d3c6f… L2M versions are V1,V2
  ✅  5e9d3c6f… L1M has versions[]
  ✅  5e9d3c6f… L1M versions are V1,V2
  ✅  5e9d3c6f… L1S has versions[]
  ✅  5e9d3c6f… L1S versions are V1,V2
  ✅  5e9d3c6f… L2S has versions[]
  ✅  5e9d3c6f… L2S versions are V1,V2
  ✅  5e9d3c6f… L3S has versions[]
  ✅  5e9d3c6f… L3S versions are V1,V2
  ✅  5e9d3c6f… L4S has versions[]
  ✅  5e9d3c6f… L4S versions are V1,V2
  ✅  5e9d3c6f… L1F has versions[]
  ✅  5e9d3c6f… L1F versions are V1,V2
  ✅  5e9d3c6f… L2F has NO versions
  ✅  5e9d3c6f… L4W has NO versions
  ✅  113738a3… has 12 practices
  ✅  113738a3… practice codes match spec
  ✅  113738a3… P0 ha